# Audio Transcription with Whisper

Advancing AI Anthropology through computational approaches to qualitative research.

[Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

<br>

---

<br>

## What This Notebook Does

This notebook transcribes audio and video recordings using OpenAI's Whisper speech recognition model, running entirely on your machine with no data sent to external servers. Upload interview recordings, lectures, or fieldwork audio and get timestamped transcripts with optional speaker identification.

The notebook uses **faster-whisper**, an optimized implementation that runs 4x faster than the original Whisper while producing identical results. Speaker diarization (identifying who is speaking) is available via **pyannote.audio** for interview transcription. All processing is local — your recordings never leave the notebook.

## Key Features

- **Local processing** — no API key required; recordings never leave your machine
- **GPU accelerated**: Automatically uses Colab's GPU when available for faster processing
- **Speaker diarization**: Optional speaker identification for interview recordings (requires a free HuggingFace token)
- **Multi-format input**: Accepts audio (.mp3, .wav, .m4a, .ogg) and video (.mp4, .mov, .webm) files
- **Multi-file batch**: Upload and transcribe multiple recordings in one session
- **Five export formats**: TXT, SRT (subtitles), CSV, Excel, and Word with speaker formatting
- **Pipeline ready**: TXT output feeds directly into the Interview Transcript Semantic Chunker

## Workflow

1. **Setup**: Install packages and load the Whisper model
2. **Configure**: Select model size, language, and diarization settings
3. **Upload**: Load audio or video files
4. **Transcribe**: Generate timestamped transcripts with optional speaker labels
5. **Review**: Examine transcripts with timestamps and audio playback
6. **Export**: Download transcripts in your preferred format

## Applications

This notebook supports any research that involves recorded speech — transcribing ethnographic interviews, focus groups, public meetings, lectures, or oral histories. The speaker diarization feature is particularly useful for multi-speaker interviews where attributing statements to specific participants matters. Output integrates directly with the Semantic Chunker for downstream coding and thematic analysis.

## Methodological Positioning

This notebook represents a **computational anthropology** approach — using machine learning to automate the mechanical aspects of transcription while preserving the researcher's role in interpretation. Whisper transcription is not perfect. Review all transcripts for accuracy, especially for specialized terminology, proper nouns, and non-English speech.

**Important**: Automated transcription is a starting point, not a finished product. Critical passages should always be verified against the original recording.

## Target Audience

Designed for anthropologists and qualitative researchers who want to transcribe interview recordings efficiently — from graduate students processing dissertation fieldwork to applied researchers working with large audio collections.

## Technical Approach

The notebook uses **faster-whisper**, a CTranslate2-optimized reimplementation of OpenAI's Whisper. Models range from tiny (39MB) to large-v3 (1.5GB). Speaker diarization uses **pyannote.audio** (pyannote/speaker-diarization-3.1), which requires a free HuggingFace account. Video files have their audio extracted automatically via ffmpeg (pre-installed on Colab).

## AI Anthropology Toolkit

This notebook is part of the [AI Anthropology Toolkit](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit), a collection of computational notebooks for AI-assisted qualitative research. Its transcripts feed the toolkit's analysis pipeline: the [Interview Transcript Semantic Chunker](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit/blob/main/notebooks/Interview_Transcript_Semantic_Chunker.ipynb) segments them, the [Qualitative Codebook Builder](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit/blob/main/notebooks/Qualitative_Codebook_Builder.ipynb) develops codes, and the [Coding and Thematic Analysis](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit/blob/main/notebooks/Coding_and_Thematic_Analysis.ipynb) notebook applies them.

## License & Attribution

This notebook is released under the [PolyForm Noncommercial License 1.0.0](https://polyformproject.org/licenses/noncommercial/1.0.0): free for noncommercial use — research, education, and nonprofit work — with attribution to Matt Artz appreciated. For commercial licensing, contact [Matt Artz](https://www.mattartz.me/).

## Citation

If you use this toolkit in your academic research, please cite:

> Artz, Matt. 2026. AI Anthropology Toolkit. Software. Zenodo. https://doi.org/10.5281/zenodo.16728812

## References

Artz, Matt. 2023. From Machine Learning to Machine Knowing: A Digital Anthropology Approach for the Machine Interpretation of Cultures. UNESCO. https://unesdoc.unesco.org/ark:/48223/pf0000384902.

Artz, Matt. 2023. "Ten Predictions for AI and the Future of Anthropology." Anthropology News, May 8. https://doi.org/10.1111/AN.1605.

Artz, Matt. 2026. "Artificial Intelligence: The AI Anthropology Lifecycle (of, by, for AI)." In Practicing Digital Ethnography, edited by Devin Proctor. Routledge. https://doi.org/10.4324/9781032672663-29.

Artz, Matt. 2026. "Multi-Agent Ethnography: Post-Conventional Anthropological Practice Through Human-AI Collaboration." Human Organization. https://doi.org/10.1080/00664677.2026.2614501.

Artz, Matt. Forthcoming. "AI Anthropology: The Future of Applied Anthropological Practice." In Routledge Handbook of Applied Anthropology, edited by Christina Wasson, Edward B. Liebow, Karine L. Narahara, Ndukuyakhe Ndlovu, and Alaka Wali. New York: Routledge.

Radford, Alec, et al. 2023. "Robust Speech Recognition via Large-Scale Weak Supervision." Proceedings of the 40th International Conference on Machine Learning. https://arxiv.org/abs/2212.04356.

## Setup and Installation

Install faster-whisper and supporting libraries. The Whisper model downloads on first use. For faster transcription, use a GPU runtime (Runtime > Change runtime type > T4 GPU).

In [ ]:
# Install required packages (numpy/pandas pinned for Colab compatibility)
!pip install -q faster-whisper pyannote.audio pydub openpyxl python-docx ipywidgets "numpy<2.2" "pandas<3"

import re
import os
import io
import json
import unicodedata
import tempfile
import numpy as np
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, HTML, clear_output, Audio
import ipywidgets as widgets

# faster-whisper
from faster_whisper import WhisperModel
print("\u2713 faster-whisper loaded")

# pydub for audio handling
from pydub import AudioSegment
print("\u2713 pydub loaded")

# python-docx for Word export
import docx
print("\u2713 python-docx loaded")

# Colab detection
IN_COLAB = False
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    pass

# GPU detection
import torch
HAS_GPU = torch.cuda.is_available()
DEVICE = 'cuda' if HAS_GPU else 'cpu'
COMPUTE_TYPE = 'float16' if HAS_GPU else 'int8'
print(f"\u2713 Device: {DEVICE} ({'GPU acceleration enabled' if HAS_GPU else 'CPU mode \u2014 consider enabling GPU runtime for faster processing'})")

# Optional: pyannote for diarization
HAVE_PYANNOTE = False
try:
    from pyannote.audio import Pipeline as PyannotePipeline
    HAVE_PYANNOTE = True
    print("\u2713 pyannote.audio loaded (speaker diarization available)")
except ImportError:
    print("\u26a0\ufe0f pyannote.audio not available \u2014 diarization will be unavailable")
    print("   Check the pip install output above for errors.")

# Utilities
def make_slug(text):
    """Filesystem-safe slug from text."""
    text = unicodedata.normalize('NFKD', text)
    text = re.sub(r'[^\w\s-]', '', text).strip().lower()
    return re.sub(r'[-\s]+', '_', text)[:50] or 'project'

def normalize_text(text):
    """Normalize unicode artifacts in exported text."""
    if not isinstance(text, str):
        return text
    replacements = {
        '\u2011': '-', '\u2010': '-', '\u2012': '-', '\u2013': '-', '\u2014': '-',
        '\u2018': "'", '\u2019': "'", '\u201c': '"', '\u201d': '"',
        '\u2026': '...', '\xa0': ' ',
    }
    for k, v in replacements.items():
        text = text.replace(k, v)
    return text

def format_timestamp(seconds):
    """Convert seconds to HH:MM:SS,mmm format."""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    ms = int((seconds % 1) * 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{ms:03d}"

def format_timestamp_short(seconds):
    """Convert seconds to [MM:SS] format."""
    minutes = int(seconds // 60)
    secs = int(seconds % 60)
    return f"[{minutes:02d}:{secs:02d}]"

def style_excel(filepath):
    """Apply branded styling to an Excel file."""
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    wb = load_workbook(filepath)
    hf = PatternFill(start_color='274C77', end_color='274C77', fill_type='solid')
    hfont = Font(bold=True, color='FFFFFF')
    tb = Border(
        left=Side(style='thin', color='E7ECEF'), right=Side(style='thin', color='E7ECEF'),
        top=Side(style='thin', color='E7ECEF'), bottom=Side(style='thin', color='E7ECEF')
    )
    for ws in wb.worksheets:
        for cell in ws[1]:
            cell.fill = hf
            cell.font = hfont
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border = tb
        for col in ws.columns:
            max_len = max(len(str(c.value or '')) for c in col)
            ws.column_dimensions[col[0].column_letter].width = min(max(max_len + 2, 10), 60)
        ws.freeze_panes = 'A2'
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(vertical='top', wrap_text=True)
                cell.border = tb
    wb.save(filepath)

print()
print(f"\u2713 Environment: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print("\U0001f3a4 Ready to configure transcription!")

## Configuration

Select the Whisper model size, language, and whether to enable speaker diarization. Larger models are more accurate but slower. Speaker diarization requires a free HuggingFace token.

In [ ]:
# Configuration

class TranscriptionConfig:
    """Configuration for audio transcription."""
    MODEL_SIZE = 'medium'
    LANGUAGE = None  # None = auto-detect
    DIARIZE = False
    HF_TOKEN = ''
    PROJECT_NAME = ''
    PROJECT_DESCRIPTION = ''

config = TranscriptionConfig()
whisper_model = None
diarization_pipeline = None

style = {'description_width': '160px'}
layout = widgets.Layout(width='500px')

def create_config_interface():
    global whisper_model, diarization_pipeline

    config_html = '<div style="background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;">'
    config_html += '<h2 style="color: #274C77; margin-top: 0;">\U0001f3a4 Audio Transcription</h2>'
    config_html += '<p><strong>Welcome!</strong> This notebook transcribes audio and video recordings using Whisper, running entirely on your machine.</p>'
    config_html += '<h3 style="color: #274C77;">\U0001f4d6 How to Use:</h3>'
    config_html += '<ol>'
    config_html += '<li><strong>Configure:</strong> Select model size and options below</li>'
    config_html += '<li><strong>Upload:</strong> Load audio or video files</li>'
    config_html += '<li><strong>Transcribe:</strong> Generate timestamped transcripts</li>'
    config_html += '<li><strong>Review:</strong> Check transcripts with audio playback</li>'
    config_html += '<li><strong>Export:</strong> Download in your preferred format</li>'
    config_html += '</ol>'
    config_html += '</div>'

    instructions = widgets.HTML(value=config_html)

    model_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\u2699\ufe0f Model Settings</h3>')

    model_input = widgets.Dropdown(
        options=[
            ('tiny (39 MB \u2014 fastest, fair accuracy)', 'tiny'),
            ('base (74 MB \u2014 fast, good accuracy)', 'base'),
            ('small (244 MB \u2014 balanced)', 'small'),
            ('medium (769 MB \u2014 recommended, excellent accuracy)', 'medium'),
            ('large-v3 (1.5 GB \u2014 best accuracy, slowest)', 'large-v3'),
        ],
        value='medium',
        description='Model size:',
        layout=layout,
        style=style
    )

    model_help = widgets.HTML(
        '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
        'Medium is recommended for fieldwork audio. Use large-v3 for difficult recordings (heavy accents, background noise). '
        'Use small or base for quick previews.</p>'
    )

    lang_input = widgets.Dropdown(
        options=[
            ('Auto-detect', 'auto'),
            ('English', 'en'),
            ('Spanish', 'es'),
            ('French', 'fr'),
            ('Portuguese', 'pt'),
            ('German', 'de'),
            ('Italian', 'it'),
            ('Japanese', 'ja'),
            ('Chinese', 'zh'),
            ('Arabic', 'ar'),
            ('Hindi', 'hi'),
            ('Russian', 'ru'),
            ('Korean', 'ko'),
            ('Dutch', 'nl'),
            ('Turkish', 'tr'),
            ('Swedish', 'sv'),
            ('Indonesian', 'id'),
            ('Swahili', 'sw'),
            ('Tagalog', 'tl'),
        ],
        value='auto',
        description='Language:',
        layout=layout,
        style=style
    )

    lang_help = widgets.HTML(
        '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
        'Whisper supports 99 languages. Specifying the language improves accuracy. '
        'Auto-detect works well for most recordings.</p>'
    )

    # Diarization section
    diarize_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f465 Speaker Diarization (Optional)</h3>')

    diarize_toggle = widgets.Checkbox(
        value=False,
        description='Enable speaker diarization',
        style={'description_width': 'auto'},
    )

    diarize_help = widgets.HTML(
        '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
        'Identifies who is speaking in multi-speaker recordings (e.g., interviews). '
        'Requires a free HuggingFace token.</p>'
    )

    hf_token_input = widgets.Password(
        value='',
        placeholder='Enter your HuggingFace token',
        description='HF Token:',
        layout=layout,
        style=style
    )

    hf_help = widgets.HTML(
        '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
        '1. Create a free account at <a href="https://huggingface.co/join" target="_blank">huggingface.co</a><br>'
        '2. Accept terms at <a href="https://huggingface.co/pyannote/speaker-diarization-3.1" target="_blank">pyannote/speaker-diarization-3.1</a><br>'
        '3. Create a token at <a href="https://huggingface.co/settings/tokens" target="_blank">Settings > Tokens</a> (read access is sufficient)</p>'
    )

    hf_box = widgets.VBox([hf_token_input, hf_help])
    hf_box.layout.display = 'none'

    def on_diarize_change(change):
        hf_box.layout.display = '' if change['new'] else 'none'
    diarize_toggle.observe(on_diarize_change, names='value')

    # Project metadata
    project_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f4c1 Project Metadata</h3>')

    project_name = widgets.Text(
        value='',
        placeholder='e.g., Field Interviews 2026',
        description='Project name:',
        layout=layout,
        style=style
    )

    project_desc = widgets.Textarea(
        value='',
        placeholder='Brief description of the recordings',
        description='Description:',
        layout=widgets.Layout(width='500px', height='60px'),
        style=style
    )

    save_btn = widgets.Button(
        description='Save & Load Model',
        style={'button_color': '#6096BA'},
        layout=widgets.Layout(width='200px', margin='20px 0')
    )

    status = widgets.Output()

    def save_config(btn):
        global whisper_model, diarization_pipeline
        config.MODEL_SIZE = model_input.value
        config.LANGUAGE = None if lang_input.value == 'auto' else lang_input.value
        config.DIARIZE = diarize_toggle.value
        config.HF_TOKEN = hf_token_input.value.strip()
        config.PROJECT_NAME = project_name.value.strip()
        config.PROJECT_DESCRIPTION = project_desc.value.strip()

        with status:
            clear_output()

            # Load Whisper model
            print(f"Loading Whisper {config.MODEL_SIZE} model ({DEVICE})...")
            try:
                whisper_model = WhisperModel(
                    config.MODEL_SIZE,
                    device=DEVICE,
                    compute_type=COMPUTE_TYPE
                )
                print(f"\u2713 Whisper {config.MODEL_SIZE} loaded on {DEVICE}")
            except Exception as e:
                print(f"\u274c Failed to load Whisper model: {e}")
                whisper_model = None
                return

            # Load diarization pipeline if requested
            if config.DIARIZE:
                if not HAVE_PYANNOTE:
                    print("\u26a0\ufe0f pyannote.audio not available \u2014 diarization disabled")
                    config.DIARIZE = False
                elif not config.HF_TOKEN:
                    print("\u26a0\ufe0f No HuggingFace token provided \u2014 diarization disabled")
                    config.DIARIZE = False
                else:
                    print("Loading speaker diarization model...")
                    try:
                        diarization_pipeline = PyannotePipeline.from_pretrained(
                            "pyannote/speaker-diarization-3.1",
                            use_auth_token=config.HF_TOKEN
                        )
                        if HAS_GPU:
                            diarization_pipeline.to(torch.device('cuda'))
                        print("\u2713 Speaker diarization model loaded")
                    except Exception as e:
                        print(f"\u26a0\ufe0f Diarization failed to load: {e}")
                        print("   Transcription will proceed without speaker labels.")
                        config.DIARIZE = False
                        diarization_pipeline = None

            print()
            lang_label = config.LANGUAGE if config.LANGUAGE else 'auto-detect'
            print(f"\u2713 Language: {lang_label}")
            print(f"\u2713 Diarization: {'enabled' if config.DIARIZE else 'disabled'}")
            if config.PROJECT_NAME:
                print(f"\u2713 Project: {config.PROJECT_NAME}")
            print()
            print("\u2713 Configuration saved! Proceed to Upload.")

    save_btn.on_click(save_config)

    display(widgets.VBox([
        instructions,
        model_header,
        model_input,
        model_help,
        lang_input,
        lang_help,
        diarize_header,
        diarize_toggle,
        diarize_help,
        hf_box,
        project_header,
        project_name,
        project_desc,
        save_btn,
        status,
    ]))

create_config_interface()

## Upload Recordings

Upload audio or video files. The notebook accepts .mp3, .wav, .m4a, .ogg (audio) and .mp4, .mov, .webm (video). Audio is automatically extracted from video files.

In [ ]:
# Upload Recordings

AUDIO_EXTENSIONS = {'.mp3', '.wav', '.m4a', '.ogg'}
VIDEO_EXTENSIONS = {'.mp4', '.mov', '.webm'}
ALL_EXTENSIONS = AUDIO_EXTENSIONS | VIDEO_EXTENSIONS

uploaded_files = []  # list of dicts: {filename, raw_bytes, audio_path, duration_s, is_video}

def get_audio_path(filename, raw_bytes):
    """Save raw bytes to a temp file and extract audio if video. Returns path to WAV file."""
    ext = '.' + filename.rsplit('.', 1)[-1].lower()
    # Save to temp file
    tmp = tempfile.NamedTemporaryFile(suffix=ext, delete=False)
    tmp.write(raw_bytes)
    tmp.close()

    if ext in VIDEO_EXTENSIONS:
        # Extract audio from video
        try:
            audio = AudioSegment.from_file(tmp.name)
            wav_path = tmp.name.rsplit('.', 1)[0] + '.wav'
            audio.export(wav_path, format='wav')
            return wav_path
        except Exception as e:
            raise RuntimeError(f"Could not extract audio from {filename}: {e}")
    elif ext == '.wav':
        return tmp.name
    else:
        # Convert to WAV for consistency
        try:
            audio = AudioSegment.from_file(tmp.name)
            wav_path = tmp.name.rsplit('.', 1)[0] + '.wav'
            audio.export(wav_path, format='wav')
            return wav_path
        except Exception:
            return tmp.name  # Fall back to original

def get_duration(audio_path):
    """Get audio duration in seconds."""
    try:
        audio = AudioSegment.from_file(audio_path)
        return len(audio) / 1000.0
    except Exception:
        return 0

def process_uploads(change):
    """Process uploaded files."""
    upload_out.clear_output()
    new_files = []

    for file_info in uploader.value:
        fname = file_info['name'] if isinstance(file_info, dict) else file_info.name
        content = file_info['content'] if isinstance(file_info, dict) else file_info.content
        raw_bytes = bytes(content)

        ext = '.' + fname.rsplit('.', 1)[-1].lower() if '.' in fname else ''
        if ext not in ALL_EXTENSIONS:
            with upload_out:
                print(f"\u26a0\ufe0f Skipping {fname} \u2014 unsupported format")
            continue

        # Skip duplicates
        if any(f['filename'] == fname for f in uploaded_files):
            continue

        try:
            audio_path = get_audio_path(fname, raw_bytes)
            duration = get_duration(audio_path)

            new_files.append({
                'filename': fname,
                'raw_bytes': raw_bytes,
                'audio_path': audio_path,
                'duration_s': duration,
                'is_video': ext in VIDEO_EXTENSIONS,
            })
        except Exception as e:
            with upload_out:
                print(f"\u26a0\ufe0f Error processing {fname}: {e}")

    uploaded_files.extend(new_files)

    with upload_out:
        if new_files:
            print(f"\u2713 Added {len(new_files)} file(s) ({len(uploaded_files)} total)")
            print()
            summary = pd.DataFrame([{
                'File': f['filename'],
                'Type': 'video' if f['is_video'] else 'audio',
                'Duration': f"{int(f['duration_s'] // 60)}:{int(f['duration_s'] % 60):02d}",
                'Size (MB)': round(len(f['raw_bytes']) / (1024 * 1024), 1),
            } for f in uploaded_files])
            display(summary)

            total_duration = sum(f['duration_s'] for f in uploaded_files)
            print(f"\n   Total duration: {int(total_duration // 60)}:{int(total_duration % 60):02d}")

upload_html = '<div style="background-color: #E7ECEF; padding: 15px; border-radius: 10px; margin: 10px 0; border-left: 5px solid #274C77;">'
upload_html += '<h3 style="color: #274C77; margin-top: 0;">\U0001f4c2 Upload Recordings</h3>'
upload_html += '<p>Supported formats:</p>'
upload_html += '<ul style="margin: 5px 0;">'
upload_html += '<li><strong>Audio:</strong> .mp3, .wav, .m4a, .ogg</li>'
upload_html += '<li><strong>Video:</strong> .mp4, .mov, .webm (audio extracted automatically)</li>'
upload_html += '</ul>'
upload_html += '</div>'

upload_instructions = widgets.HTML(value=upload_html)

uploader = widgets.FileUpload(
    accept='.mp3,.wav,.m4a,.ogg,.mp4,.mov,.webm',
    multiple=True,
    description='Select files',
    layout=widgets.Layout(width='300px')
)

uploader.observe(process_uploads, names='value')

clear_btn = widgets.Button(
    description='Clear All Files',
    style={'button_color': '#8B8C89'},
    layout=widgets.Layout(width='150px', margin='10px 0')
)

upload_out = widgets.Output()

def on_clear(_):
    uploaded_files.clear()
    upload_out.clear_output()
    with upload_out:
        print("\u2713 All files cleared.")

clear_btn.on_click(on_clear)

display(widgets.VBox([upload_instructions, uploader, clear_btn, upload_out]))

## Transcribe

Run Whisper on all uploaded recordings. If speaker diarization is enabled, speaker labels are merged with the transcript segments by timestamp overlap. Processing time depends on recording length, model size, and whether a GPU is available.

In [ ]:
# Transcribe

all_transcripts = {}  # filename -> list of segment dicts

def assign_speakers(segments, diarization_result):
    """Merge pyannote speaker labels into Whisper segments by timestamp overlap."""
    # Build speaker timeline
    speaker_timeline = []
    for turn, _, speaker in diarization_result.itertracks(yield_label=True):
        speaker_timeline.append((turn.start, turn.end, speaker))

    for seg in segments:
        seg_mid = (seg['start'] + seg['end']) / 2
        best_speaker = 'Unknown'
        best_overlap = 0

        for sp_start, sp_end, speaker in speaker_timeline:
            # Calculate overlap
            overlap_start = max(seg['start'], sp_start)
            overlap_end = min(seg['end'], sp_end)
            overlap = max(0, overlap_end - overlap_start)

            if overlap > best_overlap:
                best_overlap = overlap
                best_speaker = speaker

        seg['speaker'] = best_speaker

    # Rename speakers to Speaker 1, Speaker 2, etc.
    unique_speakers = []
    for seg in segments:
        if seg['speaker'] not in unique_speakers and seg['speaker'] != 'Unknown':
            unique_speakers.append(seg['speaker'])

    speaker_map = {sp: f'Speaker {i+1}' for i, sp in enumerate(unique_speakers)}
    speaker_map['Unknown'] = 'Unknown'

    for seg in segments:
        seg['speaker'] = speaker_map.get(seg['speaker'], seg['speaker'])

    return segments


transcribe_btn = widgets.Button(
    description='Transcribe All',
    style={'button_color': '#6096BA'},
    layout=widgets.Layout(width='200px', margin='10px 0')
)

progress_html = widgets.HTML(value='')
transcribe_out = widgets.Output()

def on_transcribe(_):
    global all_transcripts
    transcribe_out.clear_output()
    progress_html.value = ''
    all_transcripts = {}

    if whisper_model is None:
        with transcribe_out:
            print("\u26a0\ufe0f No model loaded \u2014 run the Configuration cell first.")
        return

    if not uploaded_files:
        with transcribe_out:
            print("\u26a0\ufe0f No files uploaded \u2014 run the Upload cell first.")
        return

    total = len(uploaded_files)

    with transcribe_out:
        print(f"\U0001f3a4 Transcribing {total} file(s) with Whisper {config.MODEL_SIZE}...")
        if config.DIARIZE:
            print("   Speaker diarization: enabled")
        print()

    for idx, file_info in enumerate(uploaded_files):
        fname = file_info['filename']
        audio_path = file_info['audio_path']
        duration = file_info['duration_s']
        dur_str = f"{int(duration // 60)}:{int(duration % 60):02d}"

        progress_html.value = (
            f'<span style="color: #274C77;">Transcribing {idx + 1}/{total} \u2014 '
            f'{fname} ({dur_str})</span>'
        )

        try:
            # Run Whisper
            whisper_segments, info = whisper_model.transcribe(
                audio_path,
                language=config.LANGUAGE,
                beam_size=5,
                vad_filter=True,
            )

            # Collect segments
            segments = []
            for seg in whisper_segments:
                segments.append({
                    'start': seg.start,
                    'end': seg.end,
                    'text': normalize_text(seg.text.strip()),
                    'confidence': round(np.exp(seg.avg_logprob), 3) if seg.avg_logprob else None,
                    'speaker': None,
                })

            detected_lang = info.language if hasattr(info, 'language') else 'unknown'

            # Run diarization if enabled
            if config.DIARIZE and diarization_pipeline is not None:
                progress_html.value = (
                    f'<span style="color: #274C77;">Diarizing {idx + 1}/{total} \u2014 '
                    f'{fname}</span>'
                )
                try:
                    diarization_result = diarization_pipeline(audio_path)
                    segments = assign_speakers(segments, diarization_result)
                    n_speakers = len(set(s['speaker'] for s in segments if s['speaker'] != 'Unknown'))
                except Exception as e:
                    with transcribe_out:
                        print(f"   \u26a0\ufe0f Diarization failed for {fname}: {e}")

            all_transcripts[fname] = {
                'segments': segments,
                'language': detected_lang,
                'duration_s': duration,
                'n_segments': len(segments),
                'n_speakers': len(set(s['speaker'] for s in segments if s.get('speaker') and s['speaker'] != 'Unknown')),
            }

            with transcribe_out:
                word_count = sum(len(s['text'].split()) for s in segments)
                speaker_info = ''
                if config.DIARIZE:
                    n_sp = all_transcripts[fname]['n_speakers']
                    speaker_info = f', {n_sp} speakers'
                print(f"\u2713 {fname} \u2014 {len(segments)} segments, {word_count} words{speaker_info} (lang: {detected_lang})")

        except Exception as e:
            with transcribe_out:
                print(f"\u274c {fname}: {e}")

    progress_html.value = ''

    with transcribe_out:
        print()
        print(f"\u2713 Transcription complete! {len(all_transcripts)}/{total} files processed.")
        print("   Proceed to Review or Export.")

transcribe_btn.on_click(on_transcribe)
display(widgets.VBox([transcribe_btn, progress_html, transcribe_out]))

## Review Transcripts

Review the generated transcripts with timestamps and speaker labels. Use the dropdown to switch between files. Audio playback is available in Google Colab.

In [ ]:
# Review Transcripts

if not all_transcripts:
    print("\u26a0\ufe0f Run the Transcribe cell first.")
else:
    file_dropdown = widgets.Dropdown(
        options=list(all_transcripts.keys()),
        description='File:',
        style={'description_width': '60px'},
        layout=widgets.Layout(width='500px')
    )

    review_out = widgets.Output()

    def show_transcript(change=None):
        review_out.clear_output()
        fname = file_dropdown.value
        transcript = all_transcripts[fname]
        segments = transcript['segments']

        with review_out:
            # Summary
            summary_html = f'<div style="background-color: #E7ECEF; padding: 15px; border-radius: 10px; margin: 10px 0; border-left: 5px solid #274C77;">'
            summary_html += f'<h3 style="color: #274C77; margin-top: 0;">\U0001f4c4 {fname}</h3>'
            dur = transcript['duration_s']
            summary_html += f'<p>Duration: {int(dur // 60)}:{int(dur % 60):02d} | '
            summary_html += f'Segments: {transcript["n_segments"]} | '
            summary_html += f'Language: {transcript["language"]}'
            if transcript['n_speakers'] > 0:
                summary_html += f' | Speakers: {transcript["n_speakers"]}'
            summary_html += '</p></div>'
            display(HTML(summary_html))

            # Audio playback (Colab)
            matching = [f for f in uploaded_files if f['filename'] == fname]
            if matching:
                try:
                    display(Audio(matching[0]['audio_path']))
                except Exception:
                    pass

            # Transcript display
            transcript_html = '<div style="font-family: monospace; font-size: 13px; line-height: 1.8; padding: 15px; '
            transcript_html += 'background: #FAFAFA; border-radius: 8px; max-height: 500px; overflow-y: auto;">'

            current_speaker = None
            for seg in segments:
                ts = format_timestamp_short(seg['start'])
                text = seg['text']
                speaker = seg.get('speaker')

                if speaker and speaker != current_speaker:
                    current_speaker = speaker
                    transcript_html += f'<p style="margin: 15px 0 5px 0;"><strong style="color: #274C77;">{speaker}</strong></p>'

                conf_color = '#274C77'
                if seg.get('confidence') and seg['confidence'] < 0.5:
                    conf_color = '#CC4444'

                transcript_html += f'<span style="color: #8B8C89;">{ts}</span> '
                transcript_html += f'<span style="color: {conf_color};">{text}</span><br>'

            transcript_html += '</div>'
            display(HTML(transcript_html))

            # Word count
            word_count = sum(len(s['text'].split()) for s in segments)
            print(f"\n   Total words: {word_count}")

    file_dropdown.observe(show_transcript, names='value')

    display(file_dropdown)
    show_transcript()

## Export

Export transcripts as TXT, SRT (subtitles), CSV, Excel, or Word. TXT output is formatted for direct use with the Interview Transcript Semantic Chunker.

In [ ]:
# Export

if not all_transcripts:
    print("\u26a0\ufe0f Run the Transcribe cell first.")
else:
    export_file_dropdown = widgets.Dropdown(
        options=['All files'] + list(all_transcripts.keys()),
        value='All files',
        description='File:',
        style={'description_width': '60px'},
        layout=widgets.Layout(width='500px')
    )

    export_format = widgets.Dropdown(
        options=[
            ('TXT \u2014 plain text with speakers and timestamps', 'txt'),
            ('SRT \u2014 subtitle format', 'srt'),
            ('CSV \u2014 structured segments', 'csv'),
            ('Excel (.xlsx) \u2014 styled with summary', 'excel'),
            ('Word (.docx) \u2014 formatted transcript', 'docx'),
        ],
        value='txt',
        description='Format:',
        style={'description_width': '60px'},
        layout=widgets.Layout(width='500px')
    )

    export_btn = widgets.Button(
        description='Export',
        style={'button_color': '#6096BA'},
        layout=widgets.Layout(width='200px', margin='10px 0')
    )

    export_out = widgets.Output()

    def build_txt(fname, transcript):
        """Build TXT content with speaker labels and timestamps."""
        lines = []
        current_speaker = None
        for seg in transcript['segments']:
            speaker = seg.get('speaker')
            ts = format_timestamp_short(seg['start'])

            if speaker and speaker != current_speaker:
                current_speaker = speaker
                lines.append(f'\n{speaker}:')

            lines.append(f'{ts} {seg["text"]}')
        return '\n'.join(lines).strip()

    def build_srt(fname, transcript):
        """Build SRT subtitle content."""
        lines = []
        for i, seg in enumerate(transcript['segments'], 1):
            start = format_timestamp(seg['start'])
            end = format_timestamp(seg['end'])
            text = seg['text']
            if seg.get('speaker'):
                text = f"[{seg['speaker']}] {text}"
            lines.append(f"{i}")
            lines.append(f"{start} --> {end}")
            lines.append(text)
            lines.append("")
        return '\n'.join(lines)

    def build_segments_df(fname, transcript):
        """Build DataFrame of segments."""
        rows = []
        for i, seg in enumerate(transcript['segments']):
            rows.append({
                'filename': fname,
                'segment_id': i + 1,
                'start_time': round(seg['start'], 2),
                'end_time': round(seg['end'], 2),
                'speaker': seg.get('speaker', ''),
                'text': seg['text'],
                'confidence': seg.get('confidence', ''),
            })
        return pd.DataFrame(rows)

    def build_docx(fname, transcript):
        """Build a Word document with formatted transcript."""
        doc_obj = docx.Document()
        doc_obj.add_heading(fname, level=1)

        dur = transcript['duration_s']
        meta = f"Duration: {int(dur // 60)}:{int(dur % 60):02d}"
        meta += f" | Segments: {transcript['n_segments']}"
        meta += f" | Language: {transcript['language']}"
        if transcript['n_speakers'] > 0:
            meta += f" | Speakers: {transcript['n_speakers']}"
        doc_obj.add_paragraph(meta).runs[0].font.color.rgb = docx.shared.RGBColor(0x8B, 0x8C, 0x89)

        doc_obj.add_paragraph()

        current_speaker = None
        for seg in transcript['segments']:
            speaker = seg.get('speaker')
            ts = format_timestamp_short(seg['start'])

            if speaker and speaker != current_speaker:
                current_speaker = speaker
                p = doc_obj.add_paragraph()
                run = p.add_run(f'\n{speaker}')
                run.bold = True
                run.font.color.rgb = docx.shared.RGBColor(0x27, 0x4C, 0x77)

            p = doc_obj.add_paragraph()
            ts_run = p.add_run(f'{ts}  ')
            ts_run.font.color.rgb = docx.shared.RGBColor(0x8B, 0x8C, 0x89)
            ts_run.font.size = docx.shared.Pt(9)
            p.add_run(seg['text'])

        return doc_obj

    def on_export(_):
        export_out.clear_output()
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        slug = make_slug(config.PROJECT_NAME) if config.PROJECT_NAME else 'transcript'
        fmt = export_format.value

        # Determine which files to export
        if export_file_dropdown.value == 'All files':
            files_to_export = list(all_transcripts.items())
        else:
            fname = export_file_dropdown.value
            files_to_export = [(fname, all_transcripts[fname])]

        try:
            if fmt == 'txt':
                for fname, transcript in files_to_export:
                    file_slug = make_slug(fname.rsplit('.', 1)[0])
                    filepath = f'{slug}_{file_slug}_{timestamp}.txt'
                    content = build_txt(fname, transcript)
                    with open(filepath, 'w', encoding='utf-8') as f:
                        f.write(content)
                    with export_out:
                        print(f"\u2713 {filepath}")
                        if IN_COLAB:
                            colab_files.download(filepath)

            elif fmt == 'srt':
                for fname, transcript in files_to_export:
                    file_slug = make_slug(fname.rsplit('.', 1)[0])
                    filepath = f'{slug}_{file_slug}_{timestamp}.srt'
                    content = build_srt(fname, transcript)
                    with open(filepath, 'w', encoding='utf-8') as f:
                        f.write(content)
                    with export_out:
                        print(f"\u2713 {filepath}")
                        if IN_COLAB:
                            colab_files.download(filepath)

            elif fmt == 'csv':
                filepath = f'{slug}_{timestamp}.csv'
                all_dfs = []
                for fname, transcript in files_to_export:
                    all_dfs.append(build_segments_df(fname, transcript))
                combined = pd.concat(all_dfs, ignore_index=True)
                combined.to_csv(filepath, index=False)
                with export_out:
                    print(f"\u2713 {filepath}")
                    if IN_COLAB:
                        colab_files.download(filepath)

            elif fmt == 'excel':
                filepath = f'{slug}_{timestamp}.xlsx'
                with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
                    # Summary sheet
                    summary_rows = []
                    for fname, transcript in files_to_export:
                        dur = transcript['duration_s']
                        word_count = sum(len(s['text'].split()) for s in transcript['segments'])
                        summary_rows.append({
                            'File': fname,
                            'Duration': f"{int(dur // 60)}:{int(dur % 60):02d}",
                            'Segments': transcript['n_segments'],
                            'Words': word_count,
                            'Speakers': transcript['n_speakers'],
                            'Language': transcript['language'],
                        })
                    pd.DataFrame(summary_rows).to_excel(writer, sheet_name='Summary', index=False)

                    # Per-file sheets
                    for fname, transcript in files_to_export:
                        sheet_name = make_slug(fname.rsplit('.', 1)[0])[:31]
                        df = build_segments_df(fname, transcript)
                        df.to_excel(writer, sheet_name=sheet_name, index=False)

                style_excel(filepath)
                with export_out:
                    print(f"\u2713 {filepath}")
                    if IN_COLAB:
                        colab_files.download(filepath)

            elif fmt == 'docx':
                for fname, transcript in files_to_export:
                    file_slug = make_slug(fname.rsplit('.', 1)[0])
                    filepath = f'{slug}_{file_slug}_{timestamp}.docx'
                    doc_obj = build_docx(fname, transcript)
                    doc_obj.save(filepath)
                    with export_out:
                        print(f"\u2713 {filepath}")
                        if IN_COLAB:
                            colab_files.download(filepath)

        except Exception as e:
            with export_out:
                print(f"\u274c Export error: {type(e).__name__}: {e}")

    export_btn.on_click(on_export)

    display(widgets.VBox([
        export_file_dropdown,
        export_format,
        export_btn,
        export_out,
    ]))